# RQ3: How does country of origin influence annual sales volume and customer satisfaction?

**Hypothesis:** US and Chinese manufacturers dominate sales volume, while European brands score higher on customer satisfaction.

**Methodology:** Aggregate mean annual sales and customer rating by country → one-way ANOVA → grouped bar chart (PDF) + summary table (CSV).

In [ ]:
import pandas as pd, numpy as np, os, warnings
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.stats import f_oneway
warnings.filterwarnings('ignore')
plt.rcParams.update({'font.family':'serif','font.size':11,'axes.titlesize':13,'axes.labelsize':12,'figure.dpi':300,'axes.spines.top':False,'axes.spines.right':False})

for p in ['/kaggle/input/electric-vehicle-market-and-pricing-dataset-2026/ev_market_2026.csv','ev_market_2026.csv']:
    if os.path.exists(p): df = pd.read_csv(p); break

df = df.dropna(subset=['country_of_origin','annual_sales_units','customer_rating'])
print('Shape:', df.shape, '| Countries:', df['country_of_origin'].nunique())

In [ ]:
agg = df.groupby('country_of_origin').agg(
    N=('annual_sales_units','count'),
    Mean_Sales=('annual_sales_units','mean'),
    SE_Sales=('annual_sales_units', lambda x: x.std()/np.sqrt(len(x))),
    Mean_Rating=('customer_rating','mean'),
    SE_Rating=('customer_rating', lambda x: x.std()/np.sqrt(len(x))),
).reset_index().sort_values('Mean_Sales', ascending=False)

countries = agg['country_of_origin'].tolist()
F_s, p_s = f_oneway(*[df[df['country_of_origin']==c]['annual_sales_units'].values for c in countries])
F_r, p_r = f_oneway(*[df[df['country_of_origin']==c]['customer_rating'].values for c in countries])
print(f'Sales ANOVA: F={F_s:.2f}, p={p_s:.4f} | Rating ANOVA: F={F_r:.2f}, p={p_r:.4f}')

In [ ]:
PALETTE = ['#264653','#2a9d8f','#e9c46a','#f4a261','#e76f51','#6d6875']
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
x = np.arange(len(countries))

bars1 = ax1.bar(x, agg['Mean_Sales']/1000, color=PALETTE[:len(countries)],
                yerr=agg['SE_Sales']/1000, capsize=4, error_kw={'linewidth':1.2,'ecolor':'#555'})
ax1.set_xticks(x); ax1.set_xticklabels(countries, rotation=30, ha='right')
ax1.set_ylabel('Mean Annual Sales (thousands)')
ax1.set_title(f'Annual Sales by Country\n(F={F_s:.2f}, p={p_s:.4f})', fontsize=11)
for bar, val in zip(bars1, agg['Mean_Sales']):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1.5, f'{val/1000:.0f}k', ha='center', va='bottom', fontsize=8)

bars2 = ax2.bar(x, agg['Mean_Rating'], color=PALETTE[:len(countries)],
                yerr=agg['SE_Rating'], capsize=4, error_kw={'linewidth':1.2,'ecolor':'#555'})
ax2.set_xticks(x); ax2.set_xticklabels(countries, rotation=30, ha='right')
ax2.set_ylabel('Mean Customer Rating (1–5)')
ax2.set_title(f'Customer Rating by Country\n(F={F_r:.2f}, p={p_r:.4f})', fontsize=11)
ax2.set_ylim(3.0, agg['Mean_Rating'].max()+0.3)
for bar, val in zip(bars2, agg['Mean_Rating']):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01, f'{val:.2f}', ha='center', va='bottom', fontsize=8)

for ax in [ax1, ax2]: ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
fig.suptitle('EV Market Performance by Country of Origin', fontsize=14, y=1.01)
plt.tight_layout()
fig.savefig('RQ3_Country_Sales_Rating.pdf', bbox_inches='tight', format='pdf')
plt.show(); print('Saved: RQ3_Country_Sales_Rating.pdf')

In [ ]:
out = agg.copy()
out['Mean_Sales'] = out['Mean_Sales'].round(0).astype(int)
out['SE_Sales'] = out['SE_Sales'].round(0).astype(int)
out['Mean_Rating'] = out['Mean_Rating'].round(3)
out['SE_Rating'] = out['SE_Rating'].round(3)
out.columns = ['Country','N','Mean Annual Sales','SE Sales','Mean Customer Rating','SE Rating']
out.to_csv('RQ3_Summary_Table.csv', index=False)
print('Saved: RQ3_Summary_Table.csv'); out